# NACC_DemRKL
Copied from the original project. Paths are relative to the main folder.
Original DemRKG generation and training settings are preserved.


In [ ]:
# Paths relative to the main folder; portable between Windows and Linux.
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file())
for _path in (CLINICALKG_ROOT.parent, CLINICALKG_ROOT):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
from clinicalKG.paths import PROJECT_ROOT, SHARED_PRIMEKG, raw_primekg_path, work_dir, asset_path
DATASET = 'NACC'
METHOD = 'DemRKL'
WORK_DIR = work_dir(DATASET, METHOD)
DATASET_DIR = PROJECT_ROOT / DATASET
if METHOD == "PrimeKGL":
    SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
# A notebook kernel may previously have imported the original project's modules.
for _name in list(sys.modules):
    if _name == "experiments" or _name.startswith("experiments."):
        _file = getattr(sys.modules[_name], "__file__", None)
        if _file and not Path(_file).resolve().is_relative_to(CLINICALKG_ROOT):
            del sys.modules[_name]
if "model1" in sys.modules:
    _file = getattr(sys.modules["model1"], "__file__", None)
    if _file and Path(_file).resolve().parent != WORK_DIR:
        del sys.modules["model1"]
if str(WORK_DIR) in sys.path:
    sys.path.remove(str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(CLINICALKG_ROOT / "results"))


In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

RAW_KG_PATH = str(raw_primekg_path())
SEED_CSV_PATH = str(WORK_DIR / 'NACC_Seed_DementiaHKG.csv')
OUTPUT_HKG_PATH = str(WORK_DIR / 'NACC_DementiaHKG_PrimeKG_Sub.csv')

def build_dementia_hkg():
    print("==================================================")
    print("🏗️ 阶段 2: 开始构建 DementiaHKG (外部精炼子图) - 引入二跳与动态剪枝...")
    
    try:
        seed_df = pd.read_csv(SEED_CSV_PATH)
        print(f"   ✅ 成功加载种子文件: {SEED_CSV_PATH}")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {SEED_CSV_PATH}，请先运行第一阶段的脚本。")
        return

    valid_seeds_df = seed_df[~seed_df['NACC_Feature'].str.contains("BLACKLIST")]
    target_nodes = set(valid_seeds_df['PrimeKG_Node'].unique())
    print(f"   🎯 激活白名单种子节点共: {len(target_nodes)} 个。")

    blacklist_types = set(seed_df[seed_df['NACC_Feature'] == 'BLACKLIST_TYPE']['PrimeKG_Node'])
    blacklist_rels = set(seed_df[seed_df['NACC_Feature'] == 'BLACKLIST_REL']['PrimeKG_Node'])

    print("\n   ⏳ 正在读取庞大的 PrimeKG 原始图谱，请稍候...")
    kg_df = pd.read_csv(RAW_KG_PATH, low_memory=False)

    print("\n==================================================")
    print("🚶‍♂️ 步骤 1: 提取核心 1-hop 邻居...")
    mask_1hop = (kg_df['x_name'].isin(target_nodes)) | (kg_df['y_name'].isin(target_nodes))
    subgraph_1hop = kg_df[mask_1hop].copy()

    print("\n🧼 步骤 2: 执行 1-hop 黑名单免疫清洗...")
    mask_valid_types = (~subgraph_1hop['x_type'].isin(blacklist_types)) & \
                       (~subgraph_1hop['y_type'].isin(blacklist_types))
    mask_valid_rels = ~subgraph_1hop['relation'].isin(blacklist_rels)
    clean_1hop = subgraph_1hop[mask_valid_types & mask_valid_rels].copy()
    print(f"   👉 清洗后剩余核心 1-hop 医疗边: {len(clean_1hop)} 条")

    print("\n✂️ 步骤 3: 引入 2-hop 游走并执行基于节点全局重要度的 Top-5 智能截断...")
    
    nodes_1hop = set(clean_1hop['x_name']).union(set(clean_1hop['y_name']))

    mask_2hop_x = kg_df['x_name'].isin(nodes_1hop) & (~kg_df['y_name'].isin(target_nodes))
    mask_2hop_y = kg_df['y_name'].isin(nodes_1hop) & (~kg_df['x_name'].isin(target_nodes))
    candidates_2hop = kg_df[mask_2hop_x | mask_2hop_y].copy()

    candidates_2hop = candidates_2hop[
        (~candidates_2hop['x_type'].isin(blacklist_types)) &
        (~candidates_2hop['y_type'].isin(blacklist_types)) &
        (~candidates_2hop['relation'].isin(blacklist_rels))
    ]

    candidates_2hop['source_node'] = np.where(candidates_2hop['x_name'].isin(nodes_1hop), candidates_2hop['x_name'], candidates_2hop['y_name'])
    candidates_2hop['target_node'] = np.where(candidates_2hop['x_name'].isin(nodes_1hop), candidates_2hop['y_name'], candidates_2hop['x_name'])

    print("   ⚖️ 正在计算 2-hop 节点与 NACC 核心子图的拓扑契合度...")
    
    global_degree_x = kg_df.groupby('x_name').size()
    global_degree_y = kg_df.groupby('y_name').size()
    global_degree = global_degree_x.add(global_degree_y, fill_value=0)
    candidates_2hop['target_global_degree'] = candidates_2hop['target_node'].map(global_degree).fillna(1)
    
    local_relevance = candidates_2hop.groupby('target_node')['source_node'].nunique()
    candidates_2hop['local_relevance'] = candidates_2hop['target_node'].map(local_relevance)
    
    candidates_2hop['nacc_relevance'] = candidates_2hop['local_relevance'] / np.log1p(candidates_2hop['target_global_degree'])

    candidates_2hop = candidates_2hop.sort_values(by=['source_node', 'nacc_relevance', 'target_node'], ascending=[True, False, True])
    
    pruned_2hop = candidates_2hop.groupby('source_node').head(5).drop(columns=['source_node', 'target_node', 'nacc_relevance', 'local_relevance', 'target_global_degree'])
    
    print(f"   👉 经过重要度排序与 Top-5 截断后，保留高质量 2-hop 扩展边: {len(pruned_2hop)} 条")

    print("\n==================================================")
    print("💾 步骤 4: 组装并导出最终图谱...")
    
    final_hkg = pd.concat([clean_1hop, pruned_2hop]).drop_duplicates()
    final_hkg.to_csv(OUTPUT_HKG_PATH, index=False, encoding='utf-8')
    
    print(f"   🎉 构建成功！最终 DementiaHKG 外部子图包含: {len(final_hkg)} 条边。")
    print(f"   📄 文件已保存至: {OUTPUT_HKG_PATH}")
    
if __name__ == "__main__":
    build_dementia_hkg()

In [ ]:
from pathlib import Path
import sys
_split_root = CLINICALKG_ROOT
if str(_split_root) not in sys.path:
    sys.path.insert(0, str(_split_root))
from experiments.experiment_split import split_cohort_arrays, cohort_indices
# Fixed patient split: about 64% train, 16% validation, 20% test.
# Keep this split unchanged across model variants and all five training runs.

import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings('ignore')

# ================= ⚡ 配置区 =================
# 保持你原始的文件名，绝对不改动下游模块读取的依赖
NACC_DATA_PATH = str(DATASET_DIR / 'NACC诊断.csv')
SEED_CSV_PATH = str(WORK_DIR / 'NACC_Seed_DementiaHKG.csv')
REFINED_KG_PATH = str(WORK_DIR / 'NACC_DementiaHKG_PrimeKG_Sub.csv')

OUTPUT_GRAPH_PATH = str(WORK_DIR / 'NACC-DementiaHKG-G.csv')
OUTPUT_DICT_PATH = str(WORK_DIR / 'NACC-DementiaHKG-Entities.json')

# 新增：保存数据集划分索引，供下游多模态模型使用
OUTPUT_SPLIT_PATH = str(WORK_DIR / 'NACC_Dataset_Split.csv')

MAX_SEQ_LEN = 45
PAD_TOKEN = "[PAD]"

# 定义认知指标方向
lower_is_worse = ['boston', 'mint', 'animal', 'Fwords', 'lm_imm', 'craft_imm', 'lm_del', 'craft_del', 'digitB', 'digitBL', 'digitF', 'digitFL', 'numberB', 'numberBL', 'numberF', 'numberFL']
higher_is_worse = ['trailA', 'trailB']
MIN_SAMPLE_SIZE = 30

def build_patient_mapping_nacc():
    print("==================================================")
    print("🔗 阶段 3: 构建患者动态映射与客观图谱格式化 (包含安全的数据集划分与常模机制)...")

    # 1. 提取图谱结构 (保持原逻辑)
    try:
        kg_df = pd.read_csv(REFINED_KG_PATH)
        triplets_df = kg_df[['x_name', 'relation', 'y_name']].copy()
        triplets_df.columns = ['head', 'relation', 'tail']
        
        triplets_df.to_csv(OUTPUT_GRAPH_PATH, index=False, encoding='utf-8')
        print(f"   ✅ 标准化图谱已生成: {OUTPUT_GRAPH_PATH} (共 {len(triplets_df)} 条边)")
        print("   🔒 [安全检查]: 该图谱中不存在任何患者节点，已彻底阻断 Transductive 泄露。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {REFINED_KG_PATH}，请确保上一阶段已成功运行。")
        return

    # 2. 加载映射字典
    seed_df = pd.read_csv(SEED_CSV_PATH)
    mapping_df = seed_df[seed_df['Category'] == 'Clinical_Mapping']
    nacc_to_pkg_map = dict(zip(mapping_df['NACC_Feature'], mapping_df['PrimeKG_Node']))
    print(f"\n   📊 成功加载 {len(nacc_to_pkg_map)} 条临床特征映射规则。")

    # 3. 加载完整数据集
    try:
        nacc_df = pd.read_csv(NACC_DATA_PATH, low_memory=False)
    except FileNotFoundError:
        print(f"   ❌ 找不到患者数据 {NACC_DATA_PATH}。")
        return

    # ---------------------------------------------------------
    # 核心新增区：动态划分数据集并提取训练集用于常模计算
    # ---------------------------------------------------------
    print("\n   🔀 正在执行安全的随机数据集划分 (Train: 70%, Val: 10%, Test: 20%)...")
    
    # 获取所有有效患者 ID
    def get_pid(row):
        if 'filename' in row and pd.notna(row['filename']): return str(row['filename']).strip()
        if 'ID' in row and pd.notna(row['ID']): return str(row['ID']).strip()
        return None

    nacc_df['patient_id_temp'] = nacc_df.apply(get_pid, axis=1)
    # Reuse the fixed patient split used by every downstream model.
    from experiments.experiment_split import export_manifest, training_rows
    export_manifest("NACC", OUTPUT_SPLIT_PATH)

    # 提取训练集数据
    train_df = nacc_df.loc[training_rows("NACC", nacc_df)].copy()

    # 预处理人口学分箱
    train_df['age'] = pd.to_numeric(train_df['age'], errors='coerce')
    train_df['education'] = pd.to_numeric(train_df['education'], errors='coerce')
    train_df['age_group'] = pd.cut(train_df['age'], bins=[0, 65, 75, 120], labels=['<65', '65-75', '>75'])
    train_df['edu_group'] = pd.cut(train_df['education'], bins=[0, 12, 30], labels=['<=12', '>12'])

    nacc_df['age'] = pd.to_numeric(nacc_df['age'], errors='coerce')
    nacc_df['education'] = pd.to_numeric(nacc_df['education'], errors='coerce')
    nacc_df['age_group'] = pd.cut(nacc_df['age'], bins=[0, 65, 75, 120], labels=['<65', '65-75', '>75'])
    nacc_df['edu_group'] = pd.cut(nacc_df['education'], bins=[0, 12, 30], labels=['<=12', '>12'])

    # 计算常模 (严格限制在训练集 NC 组)
    print("   🧠 正在基于训练集 (NC组) 计算人口学常模...")
    nc_df = train_df[train_df['NC'] == 1].copy()
    normative_stats = {}
    
    global_nc_stats = {}
    for col in lower_is_worse + higher_is_worse:
        if col in nc_df.columns:
            nc_df[col] = pd.to_numeric(nc_df[col], errors='coerce')
            valid_nc = nc_df[~nc_df[col].isin([88, 99, 88.0, 99.0, -4, -1])]
            global_nc_stats[col] = {'mean': valid_nc[col].mean(), 'std': valid_nc[col].std()}

    for col in lower_is_worse + higher_is_worse:
        if col in nc_df.columns:
            valid_nc = nc_df[~nc_df[col].isin([88, 99, 88.0, 99.0, -4, -1])]
            stats = valid_nc.groupby(['age_group', 'edu_group'], observed=False)[col].agg(['mean', 'std', 'count']).to_dict(orient='index')
            
            safe_stats = {}
            for group_key, stat_values in stats.items():
                if pd.notna(stat_values['count']) and stat_values['count'] >= MIN_SAMPLE_SIZE:
                    safe_stats[group_key] = stat_values
                else:
                    safe_stats[group_key] = global_nc_stats.get(col, {'mean': np.nan, 'std': np.nan})
            normative_stats[col] = safe_stats

    # ---------------------------------------------------------
    # 动态 Z-score 检查逻辑
    # ---------------------------------------------------------
    def check_abnormal(col_name, val, row_data):
        if col_name not in lower_is_worse and col_name not in higher_is_worse:
            if col_name == 'gds': return val >= 5
            return val > 0
            
        age_grp = row_data.get('age_group')
        edu_grp = row_data.get('edu_group')
        
        if col_name in normative_stats and pd.notna(age_grp) and pd.notna(edu_grp):
            group_stat = normative_stats[col_name].get((age_grp, edu_grp))
            if group_stat and pd.notna(group_stat['mean']) and pd.notna(group_stat['std']):
                mean = group_stat['mean']
                std = max(group_stat['std'], 1e-5) 
                z_score = (val - mean) / std
                
                if col_name in lower_is_worse: return z_score < -1.5
                else: return z_score > 1.5
                    
        # 经验后备阈值
        if col_name in ['boston', 'mint', 'animal', 'Fwords']: return val < 15
        elif col_name in ['lm_imm', 'craft_imm', 'lm_del', 'craft_del']: return val <= 8
        elif col_name == 'trailA': return val > 78
        elif col_name == 'trailB': return val > 273
        elif col_name in ['digitB', 'digitBL', 'digitF', 'digitFL', 'numberB', 'numberBL', 'numberF', 'numberFL']: return val < 5
        return False

    # ---------------------------------------------------------
    # 对全量数据应用常模并生成统一特征字典
    # ---------------------------------------------------------
    patient_entities = {}
    valid_mapped_patients = 0
    total_features_mapped = 0

    print(f"\n   ⏳ 正在为所有患者建立图谱实体序列 (统一输出以保持下游文件不变)...")
    
    for index, row in nacc_df.iterrows():
        patient_id = row.get('patient_id_temp')
        if not patient_id: continue

        active_nodes = []
        for col_name, val in row.items():
            if col_name in nacc_to_pkg_map:
                val_str = str(val).strip().lower()
                if pd.notna(val) and val_str not in ['', 'nan', '-4', '-1', '88', '99', '88.0', '99.0']:
                    try:
                        num_val = float(val)
                        if check_abnormal(col_name, num_val, row):
                            primekg_node = nacc_to_pkg_map[col_name]
                            active_nodes.append(primekg_node)
                            total_features_mapped += 1
                    except ValueError:
                        pass

        active_nodes = list(set(active_nodes))
        seq_len = len(active_nodes)
        
        if seq_len > MAX_SEQ_LEN:
            active_nodes = active_nodes[:MAX_SEQ_LEN]
            attention_mask = [1] * MAX_SEQ_LEN
        else:
            pad_length = MAX_SEQ_LEN - seq_len
            attention_mask = [1] * seq_len + [0] * pad_length
            active_nodes = active_nodes + [PAD_TOKEN] * pad_length

        patient_entities[patient_id] = {
            "input_nodes": active_nodes,
            "attention_mask": attention_mask
        }
        
        if seq_len > 0:
            valid_mapped_patients += 1

    with open(OUTPUT_DICT_PATH, 'w', encoding='utf-8') as f:
        json.dump(patient_entities, f, indent=4, ensure_ascii=False)

    print("\n==================================================")
    print("💾 阶段 3 任务完成！")
    print(f"   👨‍⚕️ 处理患者总数: {len(nacc_df)}")
    print(f"   🔗 成功映射患者数: {valid_mapped_patients}")
    print(f"   📈 映射特征总次: {total_features_mapped}")
    print(f"   📄 图谱序列已保存至: {OUTPUT_DICT_PATH} (文件格式与名称均未改变)")
    print("==================================================")

if __name__ == "__main__":
    build_patient_mapping_nacc()

In [ ]:
import pandas as pd
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import warnings
import os
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ================= ⚡ 配置区 =================
GRAPH_PATH = str(WORK_DIR / 'NACC-DementiaHKG-G.csv')
OUTPUT_ENT2ID = str(WORK_DIR / 'NACC-DementiaHKG-Entity2ID.json')
OUTPUT_REL2ID = str(WORK_DIR / 'NACC-DementiaHKG-Relation2ID.json')
OUTPUT_EMBED = str(WORK_DIR / 'NACC-DementiaHKG-Embeddings.npy')
OUTPUT_LOSS_HISTORY = str(WORK_DIR / 'NACC-DementiaHKG-LossHistory.json')
OUTPUT_PLOT_PATH = str(WORK_DIR / 'NACC-DementiaHKG-LossCurve.png')

# 超参数设置 (保持与ADNI实验完全一致)
EMBED_DIM = 128
NUM_EPOCHS = 150
BATCH_SIZE = 512
LR = 0.005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= 1. 统计与索引构建 =================
def analyze_graph_and_build_dicts():
    print("==================================================")
    print("📊 步骤 1: 统计图谱规模并构建 ID 映射字典...")
    
    if not os.path.exists(GRAPH_PATH):
        raise FileNotFoundError(f"❌ 找不到图谱文件: {GRAPH_PATH}")
        
    df = pd.read_csv(GRAPH_PATH)
    
    entities = sorted(list(set(df['head'].unique()) | set(df['tail'].unique())))
    relations = sorted(list(df['relation'].unique()))
    
    entity2id = {ent: idx for idx, ent in enumerate(entities)}
    relation2id = {rel: idx for idx, rel in enumerate(relations)}
    
    with open(OUTPUT_ENT2ID, 'w', encoding='utf-8') as f:
        json.dump(entity2id, f, ensure_ascii=False, indent=4)
    with open(OUTPUT_REL2ID, 'w', encoding='utf-8') as f:
        json.dump(relation2id, f, ensure_ascii=False, indent=4)
        
    print(f"   📈 [DementiaHKG 规模统计]:")
    print(f"      - 实体节点总数 (Entities): {len(entities)}")
    print(f"      - 关系类型总数 (Relations): {len(relations)}")
    print(f"      - 三元组边总数 (Triplets): {len(df)}")
    
    return df, entity2id, relation2id

# ================= 2. 数据集与模型定义 =================
class PureKGDataset(Dataset):
    def __init__(self, triplets_df, entity2id, relation2id):
        self.triplets = []
        for _, row in triplets_df.iterrows():
            h = entity2id[row['head']]
            r = relation2id[row['relation']]
            t = entity2id[row['tail']]
            self.triplets.append((h, r, t))
        self.triplets = torch.LongTensor(self.triplets)
        
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        return self.triplets[idx]

class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, embed_dim):
        super(DistMult, self).__init__()
        self.num_entities = num_entities
        self.ent_emb = nn.Embedding(num_entities, embed_dim)
        self.rel_emb = nn.Embedding(num_relations, embed_dim)
        
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)
        self.criterion = nn.MarginRankingLoss(margin=1.0)  # 定义损失函数

    def forward(self, h, r, t):
        h_e, r_e, t_e = self.ent_emb(h), self.rel_emb(r), self.ent_emb(t)
        return torch.sum(h_e * r_e * t_e, dim=1)# 前向传播定义

    def calculate_loss(self, h, r, t):
        batch_size = h.size(0)
        neg_t = torch.randint(0, self.num_entities, (batch_size,), device=h.device)
        pos_score = self.forward(h, r, t)
        neg_score = self.forward(h, r, neg_t)
        
        target = torch.ones(batch_size, device=h.device)
        return self.criterion(pos_score, neg_score, target)

# ================= 3. 训练主循环 =================
def train_kge():
    df, entity2id, relation2id = analyze_graph_and_build_dicts()
    
    print("\n==================================================")
    print(f"🚀 步骤 2: 启动 DistMult 模型训练 (设备: {DEVICE})...")
    
    dataset = PureKGDataset(df, entity2id, relation2id)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    model = DistMult(len(entity2id), len(relation2id), EMBED_DIM).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    loss_history = [] 
    
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0
        
        for batch in dataloader:
            batch = batch.to(DEVICE)
            h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
            
            optimizer.zero_grad()
            loss = model.calculate_loss(h, r, t)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(dataloader)
        loss_history.append(avg_loss)
        
        # 为了避免终端输出过长，设定每10轮打印一次Loss
        if epoch % 10 == 0 or epoch == 1:
            print(f"   - Epoch [{epoch:03d}/{NUM_EPOCHS}] | Train Loss: {avg_loss:.4f}")
            
    print("\n==================================================")
    print("💾 步骤 3: 提取并保存实体向量矩阵及 Loss 历史...")
    embeddings = model.ent_emb.weight.detach().cpu().numpy()
    np.save(OUTPUT_EMBED, embeddings)
    
    with open(OUTPUT_LOSS_HISTORY, 'w', encoding='utf-8') as f:
        json.dump(loss_history, f)
    
    print(f"   ✅ KGE 训练完成！")
    print(f"   📐 导出的向量矩阵形状: {embeddings.shape}")
    print(f"   📄 文件已保存至: {OUTPUT_EMBED} & {OUTPUT_LOSS_HISTORY}")
    print("==================================================")
    
    return loss_history

# ================= 4. 绘制Loss曲线 =================
def plot_loss_curve(losses):
    print("\n==================================================")
    print("📈 步骤 4: 正在绘制 KGE 训练 Loss 收敛曲线...")
    
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(losses) + 1), losses, color='#2c3e50', linewidth=2, label='DistMult Train Loss')
    
    plt.title('NACC DementiaHKG Knowledge Graph Embedding Loss', fontsize=16, fontweight='bold')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Margin Ranking Loss', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=12)
    plt.tight_layout()
    
    plt.savefig(OUTPUT_PLOT_PATH, dpi=300)
    print(f"   ✅ 曲线图已生成，已保存高清大图至: {OUTPUT_PLOT_PATH}")
    print("==================================================")
    plt.show()

if __name__ == "__main__":
    losses = train_kge()
    plot_loss_curve(losses)